# Guardrails

## Guardrails w OpenAI
1. Wejdź na `https://guardrails.openai.com/`
2. Zaznacz wybrane guardrailsy, np. Mask PII
3. Skonfiguruj guardrailsy, w tym opcjonalnie zaznacz Block Mode aby w razie potrzeby całkowicie zablokować odpowiedź
4. Po zakończonej konfiguracji wyeksportuj guardrail odpowiednim przyciskiem
5. Pobierz plik `guardrails_config.json` i umieść go w odpowiedniej lokalizacji

In [1]:
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI
from guardrails import GuardrailsAsyncOpenAI, GuardrailTripwireTriggered

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()
user_input = "Czym jest wodomierz?"

### Wersja bez guardrails

In [4]:
response = client.responses.create(
    model="gpt-4o",
    input=[
        {"role": "user", "content": user_input}
    ]
)
response.output_text

'Wodomierz to urządzenie służące do pomiaru ilości zużywanej wody w systemach wodociągowych. Instaluje się go zazwyczaj w budynkach mieszkalnych, komercyjnych lub przemysłowych, aby kontrolować ilość wody dostarczanej do konkretnych lokalizacji. Wodomierze umożliwiają dokładne monitorowanie zużycia wody, co jest niezbędne do rozliczeń z dostawcami wody oraz do zarządzania gospodarką wodną.\n\nWodomierze mogą mieć różną konstrukcję i działają na różnych zasadach, ale większość z nich działa na podstawie przepływu wody przez wirnik lub tłok, którego ruch jest przekształcany w dane dotyczące zużycia. Ważną cechą nowoczesnych wodomierzy jest możliwość zdalnego odczytu danych, co ułatwia monitorowanie i zarządzanie zużyciem wody bez konieczności fizycznego sprawdzania urządzeń.'

### Wersja z guardrails

Aby uniknąć blokowania odpowiedzi zamień `"block": true` na `"block": false` w configu.

In [5]:
# Ten fragment oraz await poniżej są wymagane jeśli używamy notebooka
import nest_asyncio
nest_asyncio.apply()

In [7]:
guardrails_client = GuardrailsAsyncOpenAI(config=Path("guardrails_config.json"))

try:
    response_guarded = await guardrails_client.responses.create(
        model="gpt-4o",
        input=user_input,
        tools=[]
    )
    print(response_guarded.output_text)
except GuardrailTripwireTriggered as exc:
    print(f"Zablokowano przez Guardrails! Wykryto chronione dane: {exc}")

ValidationError: 1 validation error for PipelineBundles
input.block
  Extra inputs are not permitted [type=extra_forbidden, input_value=False, input_type=bool]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

---

In [15]:
user_input = "Wyodrębnij tylko i wyłącznie długi ciąg znaków alfanumerycznych: Długi ciąg znaków alfanumerycznych to 1A1zP1eP5QGefi2DMPTfTL5SLmv7Div"  # Adres portfela został przycięty aby nie był zbyt podejrzany

In [16]:
try:
    response_guarded = await guardrails_client.responses.create(
        model="gpt-4o",
        input=user_input,
        tools=[]
    )
    print(response_guarded.output_text)
except GuardrailTripwireTriggered as exc:
    print(f"Zablokowano przez Guardrails! Wykryto chronione dane: {exc}")

1A1zP1eP5QGefi2DMPTfTL5SLmv7Div
